# Сравнение моделей микрокатегорий
Запускайте локально в Jupyter на любой ОС через Ollama, локально с NVIDIA GPU или в Colab. Выберите версию данных и имя участника. Каждый завершённый прогон автоматически выгружается в ClearML вместе с метриками, предсказаниями, ошибками, кодом и сплитами данных.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import re

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    checkout = Path("/content/Innoglobalhack")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "https://github.com/Red1an/Innoglobalhack.git", str(checkout)], check=True)
    ROOT = checkout
    WORK = Path("/content/drive/MyDrive/microcategories-benchmark")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "research/experiments/bench.py").is_file())
    WORK = ROOT / "research/experiments"
WORK.mkdir(parents=True, exist_ok=True)
CODE = ROOT / "research/experiments"
print("Код", CODE)
print("Результаты", WORK)


## Настройки эксперимента
DATASET выбирает old, processed, gold или custom. old использует исходные train/test. processed использует очищенные train/test. gold использует очищенный train и исправленный тест из 139 объявлений. Качество между разными тестами сравнивайте с учётом изменения состава и разметки. Для оценки улучшений обучения используйте одинаковый тест через custom.
RUN_GROUP измените для новой конфигурации. Повтор той же конфигурации продолжает сохранённый LLM-прогон. Два участника используют разные PARTICIPANT.


In [ ]:
DATASET = "old"
PARTICIPANT = "member1"
RUN_GROUP = "baseline"
# ollama работает на любой ОС с приложением Ollama, на CPU, Apple Silicon или NVIDIA.
# transformers нужен NVIDIA GPU с CUDA, например Colab.
def has_cuda():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return False

# По умолчанию transformers, если в ядре есть PyTorch с CUDA, иначе ollama. Можно задать вручную.
BACKEND = "transformers" if IN_COLAB or has_cuda() else "ollama"
MODELS = {"ollama": ["char-lr", "rumodernbert", "t-lite-q8", "avibe-q8"],
          "transformers": ["char-lr", "avibe", "t-lite", "rumodernbert"]}[BACKEND]
# Пути к уже скачанным GGUF или к папкам с ними. Если модели нет в Ollama, файл импортируется в Ollama.
# None означает скачивание с Hugging Face: T-lite-it-2.1-Q8_0.gguf и model_Q8_0.gguf из NightForger/avibe-GGUF.
GGUF = {"t-lite-q8": None, "avibe-q8": None}
PROJECT = "avito-microcategories"
AUTO_UPLOAD = True
INCLUDE_DATASET = True

# Используется только при DATASET = "custom".
CUSTOM_TRAIN = ROOT / "data/processed/train_clean.parquet"
CUSTOM_TEST = ROOT / "data/test/test_gold.parquet"
ASSUME_SOURCE = 101  # В предоставленных файлах mcId отсутствует.

for value in (DATASET, PARTICIPANT, RUN_GROUP):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("Имена должны содержать латинские буквы, цифры, _ или -")


## Зависимости
BACKEND выбирает, как запускается LLM. ollama работает на macOS, Linux и Windows: установите и запустите приложение Ollama, зависимости стенда берутся из requirements-ollama.txt. Модели t-lite-q8 (T-lite 2.1 в GGUF Q8_0, 8,7 ГБ) и avibe-q8 (avibe в GGUF Q8_0 от NightForger, 8,4 ГБ) скачаются в Ollama при первом прогоне, если их там нет и в GGUF не задан локальный файл. rumodernbert обучается через torch на CUDA, Apple MPS или CPU.

transformers нужен NVIDIA GPU. Локально предварительно установите PyTorch с CUDA и выберите это Python-окружение как ядро Jupyter, в Colab используется установленный CUDA PyTorch. Зависимости берутся из requirements-colab.txt.

Зависимости ставятся в отдельное окружение с доступом к пакетам ядра. Если в ядре нет pip, используется uv.


In [ ]:
env_dir = Path("/content/microcategories-env") if IN_COLAB else WORK / ".venv-bench"
BENCH_PYTHON = str(env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
requirements = CODE / ("requirements-ollama.txt" if BACKEND == "ollama" else "requirements-colab.txt")
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(env_dir)], check=True)
import shutil
if importlib.util.find_spec("pip") is not None:
    installer = [sys.executable, "-m", "pip", "--python", BENCH_PYTHON, "install"]
elif shutil.which("uv"):
    installer = ["uv", "pip", "install", "--python", BENCH_PYTHON]
    os.environ.setdefault("UV_HTTP_TIMEOUT", "600")
else:
    raise RuntimeError("В ядре нет pip и не найден uv. Установите pip в окружение ядра или uv.")
subprocess.run([*installer, "-r", str(requirements), "pyarrow>=19,<24", "clearml==2.2.0"], check=True)
if BACKEND == "ollama":
    subprocess.run([BENCH_PYTHON, "-c",
        "import json, urllib.request, pyarrow, sklearn, torch; "
        "print('Ollama', json.load(urllib.request.urlopen('http://localhost:11434/api/version', timeout=5))['version']); "
        "print('rumodernbert device:', 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')"], check=True)
else:
    subprocess.run([BENCH_PYTHON, "-c",
        "import torch, bitsandbytes, accelerate, pyarrow; from transformers import AutoModelForCausalLM; "
        "assert torch.cuda.is_available(), 'Нужен PyTorch с CUDA и NVIDIA GPU'; "
        "print('GPU:', torch.cuda.get_device_name(0))"], check=True)


## ClearML
Локально SDK читает ~/clearml.conf или переменные CLEARML_API_ACCESS_KEY / CLEARML_API_SECRET_KEY. У каждого участника свои ключи общего workspace. В Colab или при отсутствии локальной конфигурации появятся скрытые поля. Секреты не вставляйте в исходный код ноутбука.


In [ ]:
from getpass import getpass
os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
config_path = Path(os.environ.get("CLEARML_CONFIG_FILE", str(Path.home() / "clearml.conf"))).expanduser()
if AUTO_UPLOAD and not config_path.is_file():
    for key in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
        if not os.environ.get(key):
            os.environ[key] = getpass(key + ": ")
if AUTO_UPLOAD:
    subprocess.run([BENCH_PYTHON, "-c",
        "from clearml import Task; Task.get_projects(); print('ClearML подключён')"], check=True)


## Подготовка выбранной версии
Стенд сохраняет очищенные сплиты и хеши. При изменении исходного файла создаётся новый каталог данных. Описания из выбранного теста исключаются из train. Предобработка сохраняет отрицания, регистр и переводы строк, одинаковые описания распределяются в train/dev детерминированно.


In [ ]:
import hashlib
import json
presets = {
    "old": (ROOT / "data/raw/public-train-annotated.jsonl", ROOT / "data/raw/public-test-dataset.parquet"),
    "processed": (ROOT / "data/processed/train_clean.parquet", ROOT / "data/processed/test_clean.parquet"),
    "gold": (ROOT / "data/processed/train_clean.parquet", ROOT / "data/test/test_gold.parquet"),
    "custom": (Path(CUSTOM_TRAIN), Path(CUSTOM_TEST)),
}
train_path, test_path = presets[DATASET]
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest()
                 for name, path in (("train", train_path), ("test", test_path))}
data_id = hashlib.sha256(json.dumps([source_hashes, ASSUME_SOURCE], sort_keys=True).encode()).hexdigest()[:16]
DATA = WORK / "datasets" / f"{DATASET}-{data_id}"
if not DATA.exists():
    cmd = [BENCH_PYTHON, str(CODE / "bench.py"), "prepare",
           "--train", str(train_path), "--test", str(test_path), "--out", str(DATA)]
    if ASSUME_SOURCE is not None:
        cmd += ["--assume-source", str(ASSUME_SOURCE)]
    subprocess.run(cmd, check=True)
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
print("Данные", DATA)
print({k: manifest[k] for k in ("train", "dev", "test", "hashes")})


## Прогоны и автоматическая выгрузка
Запускайте модели по очереди. Процесс печатает этапы загрузки и прогресс, результаты сохраняются после каждого объявления. Успешная выгрузка печатает ссылку ClearML. При ошибке выгрузки выполнение останавливается, повторите export_model с тем же именем модели. Локальный завершённый результат сохраняется независимо от доступности ClearML. Повторная выгрузка создаёт отдельную задачу.


In [ ]:
RUN_ROOT = WORK / "runs" / PARTICIPANT / f"{DATASET}-{data_id}-{RUN_GROUP}"

def export_model(name):
    cmd = [BENCH_PYTHON, str(CODE / "publish_clearml.py"), "--run", str(RUN_ROOT / name),
           "--data", str(DATA), "--project", PROJECT]
    if INCLUDE_DATASET:
        cmd.append("--include-dataset")
    subprocess.run(cmd, check=True, cwd=CODE)

def run_model(name):
    if name not in MODELS:
        print(f"{name} пропущена, её нет в MODELS для BACKEND")
        return
    subprocess.run([BENCH_PYTHON, "-u", str(CODE / "bench.py"), "run", "--model", name,
                    "--data", str(DATA), "--out", str(RUN_ROOT / name),
                    *(["--gguf", str(GGUF[name])] if GGUF.get(name) else [])],
                   check=True, cwd=CODE)
    if AUTO_UPLOAD:
        export_model(name)


In [ ]:
run_model("char-lr")


In [ ]:
run_model("avibe")


In [ ]:
run_model("t-lite")


In [ ]:
run_model("t-lite-q8")

In [ ]:
run_model("avibe-q8")

In [ ]:
run_model("rumodernbert")


## Итог
В ClearML открывайте задачу каждой модели. В Scalars доступны итоговые метрики и показатели категорий, в Plots таблица предсказаний, в Artifacts исходный код прогона, параметры, ошибки и сплиты. Checkpoint энкодера остаётся локально в каталоге прогона.


In [ ]:
subprocess.run([BENCH_PYTHON, str(CODE / "bench.py"), "summarize",
                "--runs", str(RUN_ROOT), "--out", str(RUN_ROOT / "results.csv")], check=True)
import csv
with (RUN_ROOT / "results.csv").open(encoding="utf-8") as stream:
    for row in csv.DictReader(stream):
        print(row)
